# Data Exploration for Bike Rental Pipeline

## 1. Project Goal

This notebook explores the raw input data for the bike rental preprocessing pipeline. 

The goal is to understand the structure of the four CSV files before building the Dagster pipeline.

In [34]:
import pandas as pd

registered = pd.read_csv("../data/registered_bike_rentals.csv")
direct = pd.read_csv("../data/direct_pickup_bike_rentals.csv")
weather = pd.read_csv("../data/weather.csv")
holidays = pd.read_csv("../data/holidays.csv")

## 2. Inspect Raw Data

### 2.1 Rental records

In [35]:
#helper
def inspect_dataframe(name, df):
    """Display basic information about a DataFrame."""
    print(f"{name} shape:", df.shape)
    print(f"{name} columns:", df.columns.tolist())
    display(df.head())

In [36]:
inspect_dataframe("Registered rentals", registered)
inspect_dataframe("Direct pickup rentals", direct)

Registered rentals shape: (2672662, 4)
Registered rentals columns: ['id', 'datetime', 'user_id', 'location_id']


,id,datetime,user_id,location_id
0,1,2011-01-01 00:05:09,158,16
1,2,2011-01-01 00:05:21,262,18
2,3,2011-01-01 00:05:39,68,18
3,4,2011-01-01 00:12:05,12,9
4,5,2011-01-01 00:25:58,91,11


Direct pickup rentals shape: (620017, 4)
Direct pickup rentals columns: ['id', 'datetime', 'user_id', 'location_id']


,id,datetime,user_id,location_id
0,1,2011-01-01 00:24:04,232,2
1,2,2011-01-01 00:30:19,54,14
2,3,2011-01-01 00:39:08,201,5
3,4,2011-01-01 01:01:12,298,13
4,5,2011-01-01 01:02:37,23,14


In [37]:
registered.columns.equals(direct.columns)

True

The registered and direct rental datasets have the same columns.  
Both datasets contain individual rental events with an `id`, `datetime`, `user_id`, and `location_id`.

Since they share the same structure, they can be processed using the same logic. The `datetime` column will be used to aggregate rental events by hour.

In [38]:
registered.info()
direct.info()

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype
---  ------       -----
 0   id           int64
 1   datetime     str  
 2   user_id      int64
 3   location_id  int64
dtypes: int64(3), str(1)
memory usage: 81.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   id           620017 non-null  int64
 1   datetime     620017 non-null  str  
 2   user_id      620017 non-null  int64
 3   location_id  620017 non-null  int64
dtypes: int64(3), str(1)
memory usage: 18.9 MB


The rental `datetime` column contains timestamps with date, hour, minute, and second information.  <br>
However, the datetime columns still need to be converted from strings to datetime objects.

In [39]:
print("Registered sample timestamps:")
print(registered["datetime"].head())

print("\nDirect sample timestamps:")
print(direct["datetime"].head())

Registered sample timestamps:
0    2011-01-01 00:05:09
1    2011-01-01 00:05:21
2    2011-01-01 00:05:39
3    2011-01-01 00:12:05
4    2011-01-01 00:25:58
Name: datetime, dtype: str

Direct sample timestamps:
0    2011-01-01 00:24:04
1    2011-01-01 00:30:19
2    2011-01-01 00:39:08
3    2011-01-01 01:01:12
4    2011-01-01 01:02:37
Name: datetime, dtype: str


As we can see, the rental records are individual event records, not hourly aggregates. So it is necessary to round each timestamp to the hour and count the number of rental events per hour.

### 2.2 Weather data

In [40]:
inspect_dataframe("Weather", weather)
weather.info()

Weather shape: (17379, 7)
Weather columns: ['id', 'datetime', 'conditions', 'temperature_c', 'perceived_temperature_c', 'humidity', 'windspeed_kmh']


,id,datetime,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,1,2011-01-01 00:00:00,clear,3.3,3.0,81.0,0.0
1,2,2011-01-01 01:00:00,clear,2.3,2.0,80.0,0.0
2,3,2011-01-01 02:00:00,clear,2.3,2.0,80.0,0.0
3,4,2011-01-01 03:00:00,clear,3.3,3.0,75.0,0.0
4,5,2011-01-01 04:00:00,clear,3.3,3.0,75.0,0.0


<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       17379 non-null  int64  
 1   datetime                 17379 non-null  str    
 2   conditions               17379 non-null  str    
 3   temperature_c            17379 non-null  float64
 4   perceived_temperature_c  17379 non-null  float64
 5   humidity                 17379 non-null  float64
 6   windspeed_kmh            17379 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 950.5 KB


The weather dataset contains weather information such as conditions, temperature, perceived temperature, humidity, and wind speed.

The datetime shows that the weather dataset is hourly. This means weather data can later be merged with hourly rental demand using the hourly timestamp, as the rental data is ggregated by hour.

### 2.3 Holiday data

In [41]:
inspect_dataframe("Holidays", holidays)
holidays.info()

Holidays shape: (21, 3)
Holidays columns: ['id', 'date', 'holiday']


,id,date,holiday
0,1,2011-01-17,"Dr. Martin Luther King, Jr.'s Birthday"
1,2,2011-02-21,Washington's Birthday
2,3,2011-04-15,D.C. Emancipation Day (observed)
3,4,2011-05-30,Memorial Day
4,5,2011-07-04,Independence Day


<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       21 non-null     int64
 1   date     21 non-null     str  
 2   holiday  21 non-null     str  
dtypes: int64(1), str(2)
memory usage: 636.0 bytes


The holidays dataset contains daily records, not hourly data, so they should not be merged directly with hourly datetime values. Instead, a date column should be extracted from the rental datetime values before merging with the holidays dataset.

## 3. Check Missing Value

In [42]:
print("Registered missing values:")
print(registered.isna().sum())

print("\nDirect missing values:")
print(direct.isna().sum())

print("\nWeather missing values:")
print(weather.isna().sum())

print("\nHolidays missing values:")
print(holidays.isna().sum())

Registered missing values:
id             0
datetime       0
user_id        0
location_id    0
dtype: int64

Direct missing values:
id             0
datetime       0
user_id        0
location_id    0
dtype: int64

Weather missing values:
id                         0
datetime                   0
conditions                 0
temperature_c              0
perceived_temperature_c    0
humidity                   0
windspeed_kmh              0
dtype: int64

Holidays missing values:
id         0
date       0
holiday    0
dtype: int64


There are no missing values in the four raw datasets.

## 4. Convert Datetime Columns

In [43]:
registered["datetime"] = pd.to_datetime(registered["datetime"])
direct["datetime"] = pd.to_datetime(direct["datetime"])
weather["datetime"] = pd.to_datetime(weather["datetime"])
holidays["date"] = pd.to_datetime(holidays["date"])

registered.info()

<class 'pandas.DataFrame'>
RangeIndex: 2672662 entries, 0 to 2672661
Data columns (total 4 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   datetime     datetime64[us]
 2   user_id      int64         
 3   location_id  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 81.6 MB


In [44]:
direct.info()

<class 'pandas.DataFrame'>
RangeIndex: 620017 entries, 0 to 620016
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   id           620017 non-null  int64         
 1   datetime     620017 non-null  datetime64[us]
 2   user_id      620017 non-null  int64         
 3   location_id  620017 non-null  int64         
dtypes: datetime64[us](1), int64(3)
memory usage: 18.9 MB


In [45]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id                       17379 non-null  int64         
 1   datetime                 17379 non-null  datetime64[us]
 2   conditions               17379 non-null  str           
 3   temperature_c            17379 non-null  float64       
 4   perceived_temperature_c  17379 non-null  float64       
 5   humidity                 17379 non-null  float64       
 6   windspeed_kmh            17379 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(1)
memory usage: 950.5 KB


In [46]:
holidays.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   id       21 non-null     int64         
 1   date     21 non-null     datetime64[us]
 2   holiday  21 non-null     str           
dtypes: datetime64[us](1), int64(1), str(1)
memory usage: 636.0 bytes


The columns `datetime` in rental and weather datasets, and `date` column in holidays dataset are now coverted from `string` to `datetime64` using `pd.to_datetime()`. This allows pandas can perform time-based operations.

## 5. Check Time Ranges and Hourly Coverage
### 5.1 Overall time ranges

In [47]:
print("Registered datetime range:")
print(registered["datetime"].min(), "to", registered["datetime"].max())

print("\nDirect datetime range:")
print(direct["datetime"].min(), "to", direct["datetime"].max())

print("\nWeather datetime range:")
print(weather["datetime"].min(), "to", weather["datetime"].max())

print("\nHoliday date range:")
print(holidays["date"].min(), "to", holidays["date"].max())

Registered datetime range:
2011-01-01 00:05:09 to 2012-12-31 23:55:53

Direct datetime range:
2011-01-01 00:24:04 to 2012-12-31 23:51:18

Weather datetime range:
2011-01-01 00:00:00 to 2012-12-31 23:00:00

Holiday date range:
2011-01-17 00:00:00 to 2012-12-25 00:00:00


### 5.2 Hourly coverage of rental and weahter data

In [48]:
# round event to hourly data
registered_hours = registered["datetime"].dt.floor("h")
direct_hours = direct["datetime"].dt.floor("h")
weather_hours = weather["datetime"].dt.floor("h")

# hourly coverage
rental_hours = pd.concat([registered_hours, direct_hours]).drop_duplicates()
weather_hours = weather_hours.drop_duplicates()

# Expected range
expected_hours = pd.date_range(
    start=min(rental_hours.min(), weather_hours.min()),
    end=max(rental_hours.max(), weather_hours.max()),
    freq="h"
)

print("Expected hourly timestamps:", len(expected_hours))
print("Unique rental hours:", rental_hours.nunique())
print("Unique weather hours:", weather_hours.nunique())

Expected hourly timestamps: 17544
Unique rental hours: 17379
Unique weather hours: 17379


The overall range of rental and weather data shows that the datasets cover the same period from 2011-01-01 to 2012-12-31. The holidays dataset contains holiday dates within this period, from 2011-01-17 to 2012-12-25.

However, the number of hourly timestamps in rental and weahter datasets are slightly smaller than the expected complete hourly range. This may due to some missing hourly records at some hours and requires separate handling in later process.

## 6. Conclusion

The raw input data contains four CSV files: registered bike rentals, direct pickup rentals, weather data, and holiday data.

The registered rental dataset and the direct pickup dataset have the same structure. The rental timestamps include minutes and seconds, which shows that the rental datasets are not hourly aggregated data.

The weather dataset contains hourly weather records. Its `datetime` column covers the same overall period as the rental data, from 2011-01-01 to 2012-12-31. This means the weather data can later be merged with hourly rental demand using the hourly timestamp. 

The holidays dataset contains specific holiday dates, not a continuous daily time series. Its date column is named `date`. This dataset is supposedt to be used as a lookup table to create an `is_holiday` feature.

The initial missing value check shows no missing values in the four datasets. The datetime-related columns were originally loaded as strings, so they were converted before time-based processing. The hourly coverage check shows that the data misses some hourly timestamps inside the range.

Overall, the data exploration confirms that the raw rental event records need to be transformed into an hourly demand dataset. The final preprocessing pipeline should explicitly create a complete hourly index, aggregate rental counts by hour, enrich the data with time-based features, merge weather information, and add holiday information before saving the final dataset for model training.